# Orders & Assets

Everything here runs on the Python engine. The C++ engine handles `SimpleOrder` and
`PositionalOrder` but drops custom order and asset subclasses (see
[Engines & Parameter Sweeps](Engines_and_Parameter_Sweeps.ipynb)).

In [1]:
from dataclasses import dataclass
from decimal import Decimal

import pandas as pd

from yabte.backtest import (
    ADFI_AVAILABLE_AT_CLOSE,
    ADFI_AVAILABLE_AT_OPEN,
    ADFI_REQUIRED,
    Book,
    OHLCAsset,
    OrderSizeType,
    OrderStatus,
    PositionalOrder,
    PositionalOrderCheckType,
    SimpleOrder,
    Strategy,
    StrategyRunner,
)
from yabte.backtest.asset import Asset
from yabte.tests._helpers import generate_nasdaq_dataset

all_assets, df_all = generate_nasdaq_dataset()

# a few weeks of three stocks keeps the tables readable
symbols = ["AAPL", "GOOG", "MSFT"]
assets = [a for a in all_assets if a.name in symbols]
data = df_all.loc[:"2018-02-14", symbols]


def run(strategy, data=data, assets=assets, cash="100000", params=None):
    return StrategyRunner(
        data=data,
        assets=assets,
        strategies=[strategy],
        books=[Book(name="Main", cash=cash)],
    ).run(params)


def trades(srr):
    cols = ["ts", "asset_name", "quantity", "price", "total", "order_label"]
    th = srr.transaction_history
    return th[cols].assign(ts=th.ts.dt.date).reset_index(drop=True)


data.head(3)

AAPL                                              \
                 Open       High        Low      Close  Adj Close   
Date                                                                
2018-01-02  42.540001  43.075001  42.314999  43.064999  40.950504   
2018-01-03  43.132500  43.637501  42.990002  43.057499  40.943363   
2018-01-04  43.134998  43.367500  43.020000  43.257500  41.133553   

                              GOOG                                   \
                 Volume       Open       High        Low      Close   
Date                                                                  
2018-01-02  102223600.0  52.417000  53.347000  52.261501  53.250000   
2018-01-03  118071600.0  53.215500  54.314499  53.160500  54.124001   
2018-01-04   89738400.0  54.400002  54.678501  54.200100  54.320000   

                                      MSFT                                   \
            Adj Close    Volume       Open       High        Low      Close   
Date                                                                          
2018-01-02  53.250000  24752000  86.129997  86.309998  85.500000  85.949997   
2018-01-03  54.124001  28604000  86.059998  86.510002  85.970001  86.349998   
2018-01-04  54.320000  20092000  86.589996  87.660004  86.570000  87.110001   

                                 
            Adj Close    Volume  
Date                             
2018-01-02  80.940384  22483800  
2018-01-03  81.317093  26061400  
2018-01-04  82.032768  21912000

## When orders execute

Each timestep:

1. `on_open()` for every strategy. Current-row fields not known at the open are hidden
   from `self.data`. For `OHLCAsset` only `Open` is visible.
2. Any order sharing a `key` with a newer one is marked `REPLACED`. The rest **execute**
   against the current row, highest `priority` first. `OHLCAsset` fills at the
   midpoint of `High` and `Low`.
3. Orders still `OPEN` carry over to the next timestep.
4. `on_close()` for every strategy, with the full row visible.
5. Books accrue interest and mark positions to the close.

An order appended in `on_open` fills **the same day**. One appended in `on_close`
fills **the next day**.

## `SimpleOrder` sizing

`size_type` sets how `size` is read:

- `QUANTITY` (default): units.
- `NOTIONAL`: a cash amount, converted to units at the fill price.
- `BOOK_PERCENT`: a percentage of the book's cash, converted at the fill price.

Quantities round to the asset's `quantity_round_dp` (default 2). Negative sizes sell.
`label` is copied onto the resulting trades.

In [2]:
class SizingDemo(Strategy):
    def on_open(self):
        if len(self.data) == 1:
            self.orders.extend(
                [
                    SimpleOrder(asset_name="AAPL", size=100, label="100 units"),
                    SimpleOrder(
                        asset_name="GOOG",
                        size=10_000,
                        size_type=OrderSizeType.NOTIONAL,
                        label="$10,000",
                    ),
                    SimpleOrder(
                        asset_name="MSFT",
                        size=25,
                        size_type=OrderSizeType.BOOK_PERCENT,
                        label="25% of cash",
                    ),
                ]
            )


srr = run(SizingDemo())
trades(srr)

,ts,asset_name,quantity,price,total,order_label
0,2018-01-02,AAPL,100.00,42.69,-4269.0000,100 units
1,2018-01-02,GOOG,189.39,52.80,-9999.7920,"$10,000"
2,2018-01-02,MSFT,249.51,85.90,-21432.9090,25% of cash


Orders execute in the order they were added, so `BOOK_PERCENT` sizes off the cash left
after the first two trades. `total` is each trade's cash effect.

## `PositionalOrder`: trade to a target

`PositionalOrder` sets the position to `size` instead of adding to it. If the position
already matches, nothing trades. Otherwise the old position is closed and the new one
opened, giving two trades on a flip from long to short.

In [3]:
class TargetDemo(Strategy):
    targets = {1: 50, 2: 50, 3: -20, 4: 0}  # day number -> target position

    def on_open(self):
        if (target := self.targets.get(len(self.data))) is not None:
            self.orders.append(
                PositionalOrder(
                    asset_name="AAPL", size=target, label=f"target {target}"
                )
            )


srr = run(TargetDemo())
display(trades(srr))
print("final AAPL position:", srr.books[0].positions["AAPL"])

,ts,asset_name,quantity,price,total,order_label
0,2018-01-02,AAPL,50.00,42.69,-2134.5000,target 50
1,2018-01-04,AAPL,-50.00,43.19,2159.5000,target -20
2,2018-01-04,AAPL,-20.00,43.19,863.8000,target -20
3,2018-01-05,AAPL,20.00,43.55,-871.0000,target 0


final AAPL position: 0.00


The day 2 order didn't trade because the position was already 50. Its status is still
`COMPLETE`:

In [4]:
pd.DataFrame(
    [(o.label, o.status.name) for o in srr.orders_processed],
    columns=["order", "status"],
)

,order,status
0,target 50,COMPLETE
1,target 50,COMPLETE
2,target -20,COMPLETE
3,target 0,COMPLETE


`check_type=PositionalOrderCheckType.ZERO_POS` only trades when the position is zero.
Use it to open a position once and then leave it alone.

## Priority

Orders queued for the same timestep sort by `priority`, highest first. Selling first
frees cash for a `BOOK_PERCENT` buy:

In [5]:
class PriorityDemo(Strategy):
    def on_open(self):
        n = len(self.data)
        if n == 1:
            self.orders.append(
                SimpleOrder(
                    asset_name="AAPL",
                    size=90,
                    size_type=OrderSizeType.BOOK_PERCENT,
                    label="buy AAPL",
                )
            )
        elif n == 5:
            # switch from AAPL to GOOG: added buy-first, but the sale runs first
            self.orders.append(
                SimpleOrder(
                    asset_name="GOOG",
                    size=90,
                    size_type=OrderSizeType.BOOK_PERCENT,
                    label="buy GOOG",
                )
            )
            self.orders.append(
                PositionalOrder(
                    asset_name="AAPL", size=0, priority=10, label="sell AAPL"
                )
            )


trades(run(PriorityDemo()))

,ts,asset_name,quantity,price,total,order_label
0,2018-01-02,AAPL,2108.22,42.69,-89999.9118,buy AAPL
1,2018-01-08,AAPL,-2108.22,43.69,92108.1318,sell AAPL
2,2018-01-08,GOOG,1661.20,55.32,-91897.5840,buy GOOG


Without the higher priority, the GOOG order would have sized off the pre-sale cash,
about 10% of the book.

## Limit orders via `pre_execute_check`

`SimpleOrder.pre_execute_check(ts, trade_price)` runs before each fill attempt.
Returning `None` fills now. `OrderStatus.OPEN` retries next timestep, and
`OrderStatus.CANCELLED` drops the order.

AAPL climbed through most of January 2018, then fell hard in early February. Of the
two resting buys below, one expires and one fills:

In [6]:
@dataclass(kw_only=True)
class LimitOrder(SimpleOrder):
    limit: Decimal
    expires: pd.Timestamp | None = None

    def pre_execute_check(self, ts, trade_price):
        if self.expires is not None and ts > self.expires:
            return OrderStatus.CANCELLED
        buying = self.size > 0
        if (buying and trade_price > self.limit) or (
            not buying and trade_price < self.limit
        ):
            return OrderStatus.OPEN  # not at our price yet, wait
        return None


class LimitDemo(Strategy):
    def on_close(self):
        if len(self.data) == 1:
            close = self.data.AAPL.Close.iloc[-1]
            for dip, days in [(0.03, 14), (0.05, 60)]:
                self.orders.append(
                    LimitOrder(
                        asset_name="AAPL",
                        size=100,
                        limit=Decimal(f"{close * (1 - dip):.2f}"),
                        expires=self.ts + pd.Timedelta(days=days),
                        label=f"{dip:.0%} dip within {days} days",
                    )
                )


srr = run(LimitDemo())
display(
    pd.DataFrame(
        [(o.label, o.limit, o.status.name) for o in srr.orders_processed],
        columns=["order", "limit", "status"],
    )
)
trades(srr)

,order,limit,status
0,3% dip within 14 days,41.77,CANCELLED
1,5% dip within 60 days,40.91,COMPLETE


,ts,asset_name,quantity,price,total,order_label
0,2018-02-02,AAPL,100.00,40.86,-4086.0000,5% dip within 60 days


## Replacing orders with `key`

Among orders sharing a `key`, only the newest survives. Older ones still waiting are
marked `REPLACED`. A strategy can re-quote a resting order daily without tracking the
previous one. The limit here trails 2% under each close until it fills:

In [7]:
class TrailingEntry(Strategy):
    def on_close(self):
        if self.books["Main"].positions["MSFT"] == 0:
            close = self.data.MSFT.Close.iloc[-1]
            self.orders.append(
                LimitOrder(
                    asset_name="MSFT",
                    size=100,
                    limit=Decimal(f"{close * 0.98:.2f}"),
                    key="msft-entry",
                    label=f"limit {close * 0.98:.2f} on {self.ts.date()}",
                )
            )


srr = run(TrailingEntry())
display(
    pd.DataFrame(
        [(o.label, o.status.name) for o in srr.orders_processed],
        columns=["order", "status"],
    )
)
trades(srr)

,order,status
0,limit 84.23 on 2018-01-02,REPLACED
1,limit 84.62 on 2018-01-03,REPLACED
2,limit 85.37 on 2018-01-04,REPLACED
3,limit 86.43 on 2018-01-05,REPLACED
4,limit 86.51 on 2018-01-08,REPLACED
5,limit 86.46 on 2018-01-09,REPLACED
6,limit 86.06 on 2018-01-10,REPLACED
7,limit 86.32 on 2018-01-11,REPLACED
8,limit 87.81 on 2018-01-12,REPLACED
9,limit 86.58 on 2018-01-16,REPLACED


,ts,asset_name,quantity,price,total,order_label
0,2018-02-08,MSFT,100.00,87.32,-8732.0000,limit 87.82 on 2018-02-07


Each `REPLACED` row is a quote superseded the next day. Only one quote is live at a
time, so the position is bought once. After the fill the strategy stops quoting.

## Follow-on orders: `post_complete` and `suborders`

`post_complete(trades)` runs after an order fills. Orders it appends to
`self.suborders` are queued for the next timestep. The bracket order below uses this to
place a take-profit and a stop-loss once the entry fills. Each exit cancels itself
once the other has closed the position.

In [8]:
@dataclass(kw_only=True)
class StopOrder(SimpleOrder):
    """Sell when the price falls to `stop`; cancel if the position is gone."""

    stop: Decimal

    def pre_execute_check(self, ts, trade_price):
        if self.book.positions[self.asset_name] <= 0:
            return OrderStatus.CANCELLED
        return None if trade_price <= self.stop else OrderStatus.OPEN


@dataclass(kw_only=True)
class TakeProfitOrder(LimitOrder):
    def pre_execute_check(self, ts, trade_price):
        if self.book.positions[self.asset_name] <= 0:
            return OrderStatus.CANCELLED
        return super().pre_execute_check(ts, trade_price)


@dataclass(kw_only=True)
class BracketOrder(SimpleOrder):
    take_profit: Decimal
    stop_loss: Decimal

    def post_complete(self, trades):
        fill = trades[0]
        self.suborders.extend(
            [
                TakeProfitOrder(
                    asset_name=self.asset_name,
                    size=-fill.quantity,
                    limit=fill.price * self.take_profit,
                    label="take profit",
                ),
                StopOrder(
                    asset_name=self.asset_name,
                    size=-fill.quantity,
                    stop=fill.price * self.stop_loss,
                    label="stop loss",
                ),
            ]
        )


class BracketDemo(Strategy):
    def on_open(self):
        if len(self.data) == 1:
            for symbol in symbols:
                self.orders.append(
                    BracketOrder(
                        asset_name=symbol,
                        size=100,
                        take_profit=Decimal("1.04"),
                        stop_loss=Decimal("0.96"),
                        label="entry",
                    )
                )


srr = run(BracketDemo())
display(trades(srr))
pd.DataFrame(
    [(o.asset_name, o.label, o.status.name) for o in srr.orders_processed],
    columns=["asset", "order", "status"],
).sort_values("asset", kind="stable")

,ts,asset_name,quantity,price,total,order_label
0,2018-01-02,AAPL,100.00,42.69,-4269.0000,entry
1,2018-01-02,GOOG,100.00,52.80,-5280.0000,entry
2,2018-01-02,MSFT,100.00,85.90,-8590.0000,entry
3,2018-01-08,GOOG,-100.00,55.32,5532.0000,take profit
4,2018-01-16,AAPL,-100.00,44.44,4444.0000,take profit
5,2018-01-16,MSFT,-100.00,89.40,8940.0000,take profit


,asset,order,status
0,AAPL,entry,COMPLETE
5,AAPL,take profit,COMPLETE
6,AAPL,stop loss,CANCELLED
1,GOOG,entry,COMPLETE
3,GOOG,take profit,COMPLETE
4,GOOG,stop loss,CANCELLED
2,MSFT,entry,COMPLETE
7,MSFT,take profit,COMPLETE
8,MSFT,stop loss,CANCELLED


Each stock exits at whichever side it hits first, +4% or -4%. The other leg cancels
itself the following day.

## Custom assets

An `Asset` declares its data fields and computes two prices, the fill price for an
order and the end-of-day mark. `OHLCAsset` is one implementation. Subclass `Asset` for
anything else and implement:

| Method | Purpose |
|---|---|
| `data_fields()` | the asset's fields and when each is known (`ADFI_*` flags) |
| `check_and_fix_data(df)` | validate and tidy the asset's columns |
| `intraday_traded_price(row, size)` | the fill price for an order of `size` |
| `end_of_day_price(row)` | the price used to mark positions |

The `ADFI_*` flags are bit masks. `ADFI_AVAILABLE_AT_OPEN` exposes a field during
`on_open` and `ADFI_AVAILABLE_AT_CLOSE` during `on_close`. Fields flagged
`ADFI_REQUIRED` must be present in the data.

`BidAskAsset` below fills buys at the ask and sells at the bid, and marks positions at
the mid. Opening quotes are known at the open. Closing quotes only appear at the
close.

In [9]:
@dataclass(kw_only=True)
class BidAskAsset(Asset):
    def data_fields(self):
        at_open_and_close = ADFI_AVAILABLE_AT_OPEN | ADFI_AVAILABLE_AT_CLOSE
        return [
            ("OpenBid", at_open_and_close | ADFI_REQUIRED),
            ("OpenAsk", at_open_and_close | ADFI_REQUIRED),
            ("Bid", ADFI_AVAILABLE_AT_CLOSE | ADFI_REQUIRED),
            ("Ask", ADFI_AVAILABLE_AT_CLOSE | ADFI_REQUIRED),
        ]

    def check_and_fix_data(self, data):
        missing = set(self._get_fields(ADFI_REQUIRED)) - set(data.columns)
        if missing:
            raise ValueError(f"{self.name} is missing fields {missing}")
        if (data.Ask < data.Bid).any():
            raise ValueError(f"{self.name} has crossed quotes")
        return data

    def intraday_traded_price(self, asset_day_data, size=None):
        # orders fill at the opening quote: the ask to buy, the bid to sell
        side = "OpenAsk" if size is None or size > 0 else "OpenBid"
        return round(Decimal(asset_day_data[side]), self.price_round_dp)

    def end_of_day_price(self, asset_day_data):
        mid = (asset_day_data.Bid + asset_day_data.Ask) / 2
        return round(Decimal(mid), self.price_round_dp)


# build bid/ask quotes 10 basis points either side of the OHLC open and close
half_spread = 0.0005
ba = pd.concat(
    {
        "FX": pd.DataFrame(
            {
                "OpenBid": data.AAPL.Open * (1 - half_spread),
                "OpenAsk": data.AAPL.Open * (1 + half_spread),
                "Bid": data.AAPL.Close * (1 - half_spread),
                "Ask": data.AAPL.Close * (1 + half_spread),
            }
        )
    },
    axis=1,
)
ba.head(3)

FX                                 
              OpenBid    OpenAsk        Bid        Ask
Date                                                  
2018-01-02  42.518731  42.561271  43.043466  43.086531
2018-01-03  43.110933  43.154066  43.035970  43.079028
2018-01-04  43.113431  43.156566  43.235871  43.279128

A round trip now pays the spread on top of the price move:

In [10]:
class RoundTrip(Strategy):
    def on_open(self):
        if len(self.data) in (1, 2):
            size = 1000 if len(self.data) == 1 else -1000
            self.orders.append(SimpleOrder(asset_name="FX", size=size))


srr = run(RoundTrip(), data=ba, assets=[BidAskAsset(name="FX")])
display(trades(srr))
srr.book_history.head(3)

,ts,asset_name,quantity,price,total,order_label
0,2018-01-02,FX,1000.00,42.56,-42560.0000,None
1,2018-01-03,FX,-1000.00,43.11,43110.0000,None


Main                   
                cash      mtm     total
ts                                     
2018-01-02   57440.0  43060.0  100500.0
2018-01-03  100550.0      0.0  100550.0
2018-01-04  100550.0      0.0  100550.0

## What a strategy can see

`self.data` runs up to the current timestep. During `on_open`, current-row fields
without `ADFI_AVAILABLE_AT_OPEN` are masked as NaN, so the day's close can't leak into
an opening decision. Columns added in `init()`, such as indicators, are masked too.
They're derived from data that isn't known yet.

In [11]:
class Peek(Strategy):
    def init(self):
        self.seen = {}

    def on_open(self):
        if len(self.data) == 3:
            self.seen["on_open"] = self.data.iloc[-1]

    def on_close(self):
        if len(self.data) == 3:
            self.seen["on_close"] = self.data.iloc[-1]


aapl = [a for a in assets if a.name == "AAPL"]
ohlc = run(Peek(), data=data[["AAPL"]], assets=aapl).strategies[0].seen
fx = run(Peek(), data=ba, assets=[BidAskAsset(name="FX")]).strategies[0].seen
pd.concat(
    [pd.DataFrame(ohlc).loc["AAPL"], pd.DataFrame(fx).loc["FX"]],
    keys=["OHLCAsset", "BidAskAsset"],
)

on_open      on_close
OHLCAsset   High             NaN  4.336750e+01
            Low              NaN  4.302000e+01
            Open       43.134998  4.313500e+01
            Close            NaN  4.325750e+01
            Volume           NaN  8.973840e+07
            Adj Close        NaN  4.113355e+01
BidAskAsset OpenBid    43.113431  4.311343e+01
            OpenAsk    43.156566  4.315657e+01
            Bid              NaN  4.323587e+01
            Ask              NaN  4.327913e+01

`OHLCAsset` exposes only `Open` during `on_open`. `BidAskAsset` also exposes its
opening quotes, because its `data_fields()` flags them as available at the open.